# Principal Component Analysis 

The central idea behind PCA is: Can we find a new set of axes that captures as much of the variation in the data as possible? If we can, then we can represent each observation using its position along that new axis rather than needing all original features. 

### A Common Misconception Before we Start: What Does "Projection" Actually Mean?

The word "projection" gets used for two related but different things, and mixing them up is easy to do.

**1. The coordinate (a number).** For a unit vector $v$, the dot product $z = v^T x$ tells us *how far along the $v$ axis* the observation sits. This single number is the observation's **score**. It lives in the new coordinate system, so it is not a point in the original space and doesn't tell us anything about pixel values on its own.

**2. The projected point (a vector).** If we take that score and travel that far in the direction of $v$, we get $(v^T x)\,v$. This *is* a vector in the original space: the closest point to $x$ on the line defined by $v$.

A quick example: let $x = \begin{bmatrix} 3 \\ 1 \end{bmatrix}$ and $v = \frac{1}{\sqrt{2}}\begin{bmatrix} 1 \\ 1 \end{bmatrix}$.

- The score is $v^T x = \frac{4}{\sqrt{2}} = 2\sqrt{2}$. This is a single number, our position along the new axis.
- The projected point is $2\sqrt{2} \cdot \frac{1}{\sqrt{2}}\begin{bmatrix} 1 \\ 1 \end{bmatrix} = \begin{bmatrix} 2 \\ 2 \end{bmatrix}$. This is where that position sits back in the original 2D space.

These two ideas map directly onto the two matrix multiplications in PCA:

- $Z = X_{\text{centred}} W_k$ computes the **scores** (meaning 1).
- $Z W_k^T + \mu$ computes the **projected points** back in the original space (meaning 2), which is the approximation of the original data.

So when we say "project the data onto the principal components", we usually mean the first one: calculating the coordinates. The second is what we do when we want to *see* what information those coordinates retained.

(Both formulas assume $v$ has length 1. If it doesn't, you'd need to divide by $v^T v$.)

## Step 1: Find the Direction that maximises variance. 

Centre the data - we want to find the maximum variance. If you don't centre the data you start measuring the distance from the origin rather than distance from the mean. 

Take one of the observations in our dataset:
$$
\mathbf{x} = 
\begin{bmatrix}
x_1 \\ 
x_2 
\end{bmatrix}
$$

Now lets say we take a new direction in our 2D space. We represent this direction using a unit vector: 

$$
\mathbf{v} = 
\begin{bmatrix}
v_1 \\ 
v_2 
\end{bmatrix}
$$

We can *project* our observations onto this new direction: 

$$
z_i = v^T x_i 
$$

This is the dot product, so we're measuring how far along our observation is on the new axis. 

Our original observations are now represented by scalar values, $z_i$. We want to choose $v$ such that the variance of these projected values is maximised (to capture as much information as possible). 

$$
Var(\bar{z}) = \frac{1}{n} \sum_{i=1}^n {\bar{z}_i^2}
$$

Substituting our projection equation: 

$$
= \frac{1}{n} \sum_{i=1}^n (v^T x_i)^2
$$

We can rearrange this using matrix notation 

$$
= \frac{1}{n} \sum_{i=1}^n
v^T x_i x_i^T v
$$

The $v$ does not change between observations, so we can take it outside the summation: 

$$
= v^T
\left(
\frac{1}{n} \sum_{i=1}^n
x_i x_i^T
\right)
v
$$

The part inside the brackets is the covariance matrix of the centred data, $C = \frac{1}{n} X^T X$.

This tells us that the variance of our data when projected onto direction $v$ is determined by the covariance matrix and our chosen direction. 

Our optimisation problem is now to choose $v$ to maximise the value of $v^TCv$, subject to $v^Tv=1$.

To solve the constrained optimisation problem, construct the Lagrangian: 

$$
L(v, \lambda) = 
v^T C v - 
\lambda(v^Tv - 1)
$$

We have that: 

$$
\frac{
    \partial L
}{\partial v} = 
2Cv - 2 \lambda v
$$

Rearranging, we have the solution of our maximal variance constraint optimisation problem: 
$$
Cv = \lambda v
$$.

And we have the definition of an eigenvector and its corresponding eigenvalue! $v$ is the eigenvector corresponding to eigenvalue, $\lambda$.

Solving this equation is equivalent to calculating the first principal component, the second principal component maximises variance such that it is orthogonal to the first principal component. 

## Transforming the Data 
In order to obtain our dimensionally reduced data we construct a matrix containing our selected eigenvectors as columns, $W$. Then our transformed matrix is $Z = XW$. 

## Work Flow for Completing this in Python
1. Start with the data matrix, $X$. 
2. Centre the data. 
3. Calculate the covariance matrix. 
4. Calculate the eigenvectors and eigenvalues. 
5. Sort the eigenvalues in descending order. 
6. Select the first *k* eigenvectors. 
7. Project the centred original data onto the new axes. 

### A Vector of Scores
You see in literature when discussing PCA a term known as the vector of scores. The vector of scores tells us where an original observation lies along each of the principal component directions.  

If you are working in NumPy, eigenvectors are stored as columns. To calculate scores of an observation, we take its centred coordinates and project them onto each eigenvector using the dot product. 



### The Dimensionally Reduced Image
scores = X_centred @ eigenvectors - these are the coordinates of each image in the new PCA coordinate system.  
X_reconstructed = scores @ eigenvectors.T + mean - if you want to display an approximation of the original data (for example, reconstructing an image)


The first multiplication projects the data onto the principal components. The second multiplication uses those same principal components to map the data back into the original coordinate system. 

The scores, tells us how far to travel along each new principal component axis. But they don't tel us the original pixel values directly. 

## Classification

The basis vectors $u_j$ are called principal components, and the vector of scores
$$
S_rx = ((x - \mu)^T u_1, ..., (x - \mu)^T u_r ) \in \mathbb{R}^r
$$

where $\mu = \frac{1}{n}\sum_{i=1}^n x_i$, the mean of the original training data. 

Classification of an image $x \in \mathbb{R}^p$ to one of the labels in {0,...,9} proceeds by assigning the label $Y_i$ with the smallest 2-norm: 

$$
||S_r (x - x_i) || _2
$$.

This is really nearest neighbour in PCA space.